In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, get_json_object, lit

spark = SparkSession.builder.getOrCreate()

# Bronze Telemetry se saare required columns extract karo
telemetry = spark.readStream.table(f"{BRONZE}.telemetry").select(
    col("event_id"), 
    col("machine_id"), 
    col("sensor_ts"),
    col("engine_temp").cast("double").alias("engine_temp"), 
    col("hydraulic_press").cast("double").alias("hydraulic_press"), 
    col("rpm").cast("long").alias("rpm"),
    col("fuel_level").cast("double").alias("fuel_level"),
    col("fault_code").cast("string").alias("fault_code"),
    lit("json").alias("_source")
)

# Kafka data (fuel_level/fault_code wahan NULL aayenge, par error nahi dega)
kafka = spark.readStream.table(f"{BRONZE}.kafka_stream").select(
    col("stream_event_id").alias("event_id"), 
    col("machine_id"), 
    col("sensor_ts"),
    get_json_object("payload_json", "$.temp").cast("double").alias("engine_temp"),
    get_json_object("payload_json", "$.hyd").cast("double").alias("hydraulic_press"),
    get_json_object("payload_json", "$.rpm").cast("long").alias("rpm"),
    lit(None).cast("double").alias("fuel_level"),
    lit(None).cast("string").alias("fault_code"),
    lit("kafka").alias("_source")
)

def dedup_telemetry(microBatchDF, batchId):
    if microBatchDF.count() == 0: return
    microBatchDF.dropDuplicates(["event_id"]).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{SILVER}.telemetry_clean")

telemetry.unionByName(kafka).writeStream.foreachBatch(dedup_telemetry).option("checkpointLocation", f"{CHECKPOINTS}/silver_telemetry_v2").trigger(availableNow=True).start().awaitTermination()

print("✅ silver_06_telemetry done (fuel_level + fault_code ke saath)!")